# Post-Delisting Padding Removal

Datastream does not terminate the series of a delisted security: it **repeats the
last valid observation** forward until the end of the requested window. Left
untreated, this generates spurious zero returns after delisting, deflates estimated
volatility, and reintroduces exactly the survivorship-related distortion that the
inclusion of delisted securities is meant to avoid.

## Method

The total return index (`RI`) is used as the reference series: its last genuine
change identifies the point beyond which the series is padded. All other market
datatypes for that security are truncated at the same date.

Using `RI` rather than each datatype separately is essential. Variables such as
shares outstanding (`NOSH`) or dividend yield (`DY`) legitimately remain constant for
long stretches — a constant value there is information, not padding — so a
per-datatype rule would truncate valid observations.

Two refinements, both established empirically on the retrieved data:

- **Non-positive `RI` values are treated as missing.** Datastream occasionally resets
  a delisted series to zero years after the event and pads at zero thereafter; taking
  the last change without this filter would place the truncation point at the reset
  rather than at the delisting.
- **Securities whose series never changes** within the sample window were already
  delisted before it begins, and are removed entirely.

## Validation

Where the delisting date could be parsed from the series name, the truncation point
is compared against it and the distribution of discrepancies is reported.

## Accounting data

Worldscope series are **not** padded: they terminate at the last reported fiscal
year. This is verified below rather than assumed.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Thesis/data"

monthly = pd.read_parquet(os.path.join(DATA_DIR, "raw_monthly.parquet"))
weekly  = pd.read_parquet(os.path.join(DATA_DIR, "raw_weekly.parquet"))
annual  = pd.read_parquet(os.path.join(DATA_DIR, "raw_annual.parquet"))
meta    = pd.read_parquet(os.path.join(DATA_DIR, "security_meta.parquet"))

for nm, d in [("monthly", monthly), ("weekly", weekly), ("annual", annual)]:
    print(f"{nm:8s} {len(d):>12,} rows | {d['symbol'].nunique():>6,} securities")
print(f"{'meta':8s} {len(meta):>12,} rows | "
      f"{int(meta['delist_date'].notna().sum()):,} with delisting date")

monthly    13,600,077 rows |  6,896 securities
weekly      7,399,302 rows |  6,350 securities
annual      1,028,439 rows |  5,200 securities
meta            7,132 rows | 3,836 with delisting date


## 2. Truncation points

For each security, the last date on which `RI` takes a value different from the
preceding one.

In [3]:
def last_real_date(df, datatype="RI"):
    """Last date on which the reference series genuinely changes value."""
    d = df[(df["datatype"] == datatype) & (df["value"] > 0)]
    d = d.sort_values(["symbol", "date"])
    prev = d.groupby("symbol")["value"].shift()
    changed = prev.isna() | (d["value"] != prev)
    return d.loc[changed].groupby("symbol")["date"].max().rename("last_real")


def first_date(df, datatype="RI"):
    d = df[(df["datatype"] == datatype) & (df["value"] > 0)]
    return d.groupby("symbol")["date"].min().rename("first_obs")


cut_m = last_real_date(monthly).to_frame().join(first_date(monthly))
cut_w = last_real_date(weekly).to_frame().join(first_date(weekly))

print(f"Monthly: truncation point for {len(cut_m):,} securities "
      f"(of {monthly['symbol'].nunique():,} with any monthly data)")
print(f"Weekly : truncation point for {len(cut_w):,} securities")

no_ri = set(monthly["symbol"]) - set(cut_m.index)
print(f"\nSecurities with monthly data but no usable RI: {len(no_ri):,}")

Monthly: truncation point for 6,342 securities (of 6,896 with any monthly data)
Weekly : truncation point for 6,348 securities

Securities with monthly data but no usable RI: 554


### Securities that are padding throughout

If the reference series never changes within the window, the security was already
delisted before the sample begins.

In [4]:
all_pad_m = cut_m.index[cut_m["last_real"] == cut_m["first_obs"]]
all_pad_w = cut_w.index[cut_w["last_real"] == cut_w["first_obs"]]

print(f"Monthly: {len(all_pad_m):,} securities constant throughout ({100*len(all_pad_m)/len(cut_m):.1f}%)")
print(f"Weekly : {len(all_pad_w):,} securities constant throughout ({100*len(all_pad_w)/len(cut_w):.1f}%)")

chk = meta[meta["symbol"].isin(all_pad_m) & meta["delist_date"].notna()]
if len(chk):
    print(f"\nOf these, {len(chk):,} have a known delisting date:")
    print(f"  median delisting year: {int(chk['delist_date'].dt.year.median())}")
    print(f"  delisted before 1996 : {int((chk['delist_date'] < '1996-01-01').sum()):,} "
          f"({100*(chk['delist_date'] < '1996-01-01').mean():.0f}%)")

Monthly: 691 securities constant throughout (10.9%)
Weekly : 660 securities constant throughout (10.4%)

Of these, 516 have a known delisting date:
  median delisting year: 1994
  delisted before 1996 : 363 (70%)


## 3. Validation against known delisting dates

Discrepancy between the truncation point and the delisting date recorded in the
series name, in months.

In [5]:
v = (meta[meta["delist_date"].notna()]
     .merge(cut_m, left_on="symbol", right_index=True, how="inner"))
v["diff_m"] = ((v["last_real"] - v["delist_date"]).dt.days / 30.44).round()

print(f"Securities validated: {len(v):,}\n")
print(v["diff_m"].describe().round(1).to_string())

bins = pd.cut(v["diff_m"], [-np.inf, -24, -6, -1, 1, 6, 24, np.inf],
              labels=["< -24m", "-24..-6m", "-6..-1m", "within 1m",
                      "1..6m", "6..24m", "> 24m"])
print("\nDistribution of (truncation - delisting):")
print(bins.value_counts().sort_index().to_string())
print(f"\nWithin 1 month : {(v['diff_m'].abs() <= 1).mean()*100:.1f}%")
print(f"Within 6 months: {(v['diff_m'].abs() <= 6).mean()*100:.1f}%")

Securities validated: 3,796

count    3796.0
mean       -0.9
std        21.5
min      -191.0
25%        -2.0
50%         0.0
75%         0.0
max       203.0

Distribution of (truncation - delisting):
diff_m
< -24m        222
-24..-6m      430
-6..-1m       789
within 1m    1989
1..6m          29
6..24m        118
> 24m         219

Within 1 month : 62.2%
Within 6 months: 75.3%


**Reading the tails.** Truncation *earlier* than the recorded delisting corresponds
to securities suspended or no longer trading well before formal removal; truncating
at the last genuine price movement is the intended behaviour. Truncation *later*
occurs almost exclusively for securities delisted before 1996, whose series is padded
from the first month of the window and which are removed by the rule above.

In [6]:
print("Truncation much earlier than delisting (< -24m):")
o = v[v["diff_m"] < -24].nsmallest(5, "diff_m")
print(o[["symbol", "delist_date", "last_real", "diff_m"]].to_string(index=False)
      if len(o) else "  none")

print("\nTruncation later than delisting (> 6m):")
o2 = v[v["diff_m"] > 6].nlargest(5, "diff_m")
print(o2[["symbol", "delist_date", "last_real", "diff_m"]].to_string(index=False)
      if len(o2) else "  none")
if len(o2):
    pre96 = (o2["delist_date"] < "1996-01-01").mean()
    print(f"  of which delisted before 1996: {100*pre96:.0f}%")

Truncation much earlier than delisting (< -24m):
symbol delist_date  last_real  diff_m
F:MLPV  2026-05-15 2010-06-01  -191.0
775769  2011-02-01 1996-01-01  -181.0
 I:BIE  2025-04-23 2010-05-01  -180.0
 F:BTP  2009-04-29 1996-03-01  -158.0
307330  2008-03-03 1996-01-01  -146.0

Truncation later than delisting (> 6m):
symbol delist_date  last_real  diff_m
929543  1979-01-18 1996-01-01   203.0
904719  1984-03-05 1996-01-01   142.0
916474  1984-03-05 1996-01-01   142.0
929284  1984-03-05 1996-01-01   142.0
929391  1984-03-05 1996-01-01   142.0
  of which delisted before 1996: 100%


## 4. Apply the truncation

Observations after the truncation point are removed, for every datatype of the
security. Securities that are padding throughout are dropped.

Securities without a usable `RI` cannot be positioned this way; where a delisting
date is known it is used instead, otherwise the series is left untouched and
flagged.

In [7]:
def apply_truncation(df, cut, all_pad, meta, label):
    n0, s0 = len(df), df["symbol"].nunique()

    # securities that are padding throughout
    df = df[~df["symbol"].isin(all_pad)].copy()
    n_after_drop = len(df)

    # fallback for securities without a reference series
    fallback = (meta.loc[meta["delist_date"].notna(), ["symbol", "delist_date"]]
                .set_index("symbol")["delist_date"].rename("last_real"))
    cutoff = cut["last_real"].copy()
    missing = set(df["symbol"]) - set(cutoff.index)
    add = fallback[fallback.index.isin(missing)]
    n_fallback = len(add)
    cutoff = pd.concat([cutoff, add])

    df["_cut"] = df["symbol"].map(cutoff)
    n_nocut = int(df["_cut"].isna().sum())
    keep = df["_cut"].isna() | (df["date"] <= df["_cut"])
    df = df.loc[keep].drop(columns="_cut")

    print(f"=== {label} ===")
    print(f"  rows before                : {n0:>12,}")
    print(f"  dropped (padding throughout): {n0-n_after_drop:>12,}  "
          f"({len(all_pad):,} securities)")
    print(f"  dropped (post-delisting pad): {n_after_drop-len(df):>12,}")
    print(f"  rows after                 : {len(df):>12,}  "
          f"({100*len(df)/n0:.1f}% retained)")
    print(f"  securities: {s0:,} -> {df['symbol'].nunique():,}")
    print(f"  truncation point from delisting date (fallback): {n_fallback:,} securities")
    print(f"  left untruncated (no reference, no date)       : "
          f"{df.loc[df['symbol'].isin(set(df['symbol'])-set(cutoff.index)), 'symbol'].nunique():,} securities")
    return df


monthly_c = apply_truncation(monthly, cut_m, all_pad_m, meta, "MONTHLY")
print()
weekly_c  = apply_truncation(weekly,  cut_w, all_pad_w, meta, "WEEKLY")

=== MONTHLY ===
  rows before                :   13,600,077
  dropped (padding throughout):    1,586,085  (691 securities)
  dropped (post-delisting pad):    5,281,538
  rows after                 :    6,732,454  (49.5% retained)
  securities: 6,896 -> 6,185
  truncation point from delisting date (fallback): 40 securities
  left untruncated (no reference, no date)       : 514 securities

=== WEEKLY ===
  rows before                :    7,399,302
  dropped (padding throughout):      959,216  (660 securities)
  dropped (post-delisting pad):    3,270,060
  rows after                 :    3,170,026  (42.8% retained)
  securities: 6,350 -> 5,689
  truncation point from delisting date (fallback): 2 securities
  left untruncated (no reference, no date)       : 0 securities


## 5. Effect on returns

The padding manifests as runs of exactly zero return. Their share before and after
cleaning is the most direct check that the procedure works.

In [8]:
def zero_return_share(df, label):
    d = df[(df["datatype"] == "RI") & (df["value"] > 0)].sort_values(["symbol", "date"])
    r = d.groupby("symbol")["value"].pct_change()
    tot, zer = r.notna().sum(), (r == 0).sum()
    print(f"  {label:9s} {zer:>10,} / {tot:>12,} returns exactly zero  ({100*zer/tot:.2f}%)")

print("Share of exactly-zero monthly returns:")
zero_return_share(monthly, "before")
zero_return_share(monthly_c, "after")

Share of exactly-zero monthly returns:
  before     1,026,301 /    1,688,086 returns exactly zero  (60.80%)
  after         63,776 /      725,561 returns exactly zero  (8.79%)


## 6. Accounting data: verification

Worldscope items should terminate at the last reported fiscal year rather than being
padded. Verified by comparing, for delisted securities, the final year of accounting
data with the delisting date.

In [9]:
ta = annual[annual["datatype"] == "WC02999"]
last_a = ta.groupby("symbol")["date"].max().rename("last_annual")
va = meta[meta["delist_date"].notna()].merge(last_a, left_on="symbol",
                                             right_index=True, how="inner")
va["last_year"] = pd.to_numeric(va["last_annual"], errors="coerce")
va["delist_year"] = va["delist_date"].dt.year
va["gap"] = va["last_year"] - va["delist_year"]

print(f"Delisted securities with accounting data: {len(va):,}\n")
print("Final accounting year minus delisting year:")
print(va["gap"].value_counts().sort_index().head(12).to_string())
print(f"\nEnding at or before the delisting year: {(va['gap'] <= 0).mean()*100:.1f}%")
print("A share close to 100% confirms that accounting series are not padded.")

Delisted securities with accounting data: 2,465

Final accounting year minus delisting year:
gap
-24     1
-23     1
-21     2
-20     3
-19     2
-17     2
-16     2
-15     5
-14     6
-13    12
-12    11
-11     9

Ending at or before the delisting year: 93.4%
A share close to 100% confirms that accounting series are not padded.


## 7. Save

In [10]:
monthly_c.to_parquet(os.path.join(DATA_DIR, "clean_monthly.parquet"), index=False)
weekly_c.to_parquet(os.path.join(DATA_DIR, "clean_weekly.parquet"), index=False)

trunc = cut_m.copy()
trunc["is_all_padding"] = trunc.index.isin(all_pad_m)
trunc.reset_index().to_parquet(os.path.join(DATA_DIR, "truncation_points.parquet"),
                               index=False)

for f in ["clean_monthly.parquet", "clean_weekly.parquet", "truncation_points.parquet"]:
    p = os.path.join(DATA_DIR, f)
    d = pd.read_parquet(p)
    print(f"  {f:28s} {len(d):>12,} rows  ({os.path.getsize(p)/1e6:.1f} MB)")

print("\nAnnual data require no cleaning and are used as retrieved.")

  clean_monthly.parquet           6,732,454 rows  (15.1 MB)
  clean_weekly.parquet            3,170,026 rows  (14.2 MB)
  truncation_points.parquet           6,342 rows  (0.1 MB)

Annual data require no cleaning and are used as retrieved.


In [13]:
u = pd.read_csv(os.path.join(DATA_DIR, "master_universe.csv"), dtype=str)
untrunc = set(monthly_c['symbol']) - set(cut_m.index) - set(meta.loc[meta['delist_date'].notna(),'symbol'])
d = u[u['Symbol'].isin(untrunc)]
print(f"Left untruncated: {len(untrunc):,}")
print(d['Activity'].value_counts())
print(d['country'].value_counts())

# quali datatype hanno? e quanto sono lunghe le loro serie?
sub = monthly_c[monthly_c['symbol'].isin(untrunc)]
print("\ndatatypes:", sub.groupby('datatype')['symbol'].nunique().to_dict())
print("mesi per titolo:", sub.groupby('symbol')['date'].nunique().describe().round(0).to_dict())

Left untruncated: 514
Activity
Dead      467
Active     47
Name: count, dtype: int64
country
ES    323
DE     75
IT     59
FR     57
Name: count, dtype: int64

datatypes: {'AF': 110, 'DPS': 103, 'DY': 5, 'EPS1MN': 406, 'EPS1SD': 393, 'MV': 4, 'NOSH': 109, 'P': 7, 'VO': 5}
mesi per titolo: {'count': 514.0, 'mean': 248.0, 'std': 115.0, 'min': 1.0, '25%': 166.0, '50%': 274.0, '75%': 360.0, 'max': 360.0}


In [14]:
has_ri = set(monthly_c.loc[monthly_c['datatype']=='RI','symbol'])
print(f"Untruncated securities with RI: {len(untrunc & has_ri)}")   # atteso: 0

Untruncated securities with RI: 0
